<h2><b>计算机高等教育通用教材</b></h2>
<h2>机器学习 Machine learning</h2>
<hr>
<h5>第一部分：监督学习 supervised learning</h5>
<h5>第五章：概率分类与朴素贝叶斯 Naive Bayes</h5>
<hr>
<h3><b>实验五：基于贝叶斯模型的金融欺诈短信识别 (Fraud Detection)</b></h3>
<hr>
<a href='https://scikit-learn.org/stable/modules/generated/sklearn.naive_bayes.MultinomialNB.html'>查看MultinomialNB源代码(sklearn)</a><br>
<br>

> **适合人群** ：前面几章，我们分别学习了“画线分割”（线性/逻辑回归）和“提问分叉”（决策树）。今天，我们要换一个全新的视角——**“上帝掷骰子”（概率统计）**。
> 本实验将带你处理人类世界中最复杂的数据格式：**文本（Text）**。机器只认识数字不认识汉字，怎么把一条短信变成一堆数字？看完这章，你不仅能手捏一个欺诈识别模型，还能彻底掌握自然语言处理（NLP）的入门基石。
<hr>

#### 第0步：测试python与虚拟环境

In [ ]:
print("Hello Naive Bayes and Fraud Detection!")
import pip
print("Pip version:", pip.__version__)

<hr><hr>

#### 第一步：import库 & 导入数据
<hr>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from sklearn import model_selection
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [ ]:
# 教材原定从外部读取大批量的短信文件。为了让你拿到代码直接能跑，
# 我们在这里手动构建一个迷你的、已经做好初步“中文分词（用空格隔开）”的短信数据集。
# 真实场景中，你会使用 jieba 等分词库来完成“把句子切成词”的步骤。

texts = [
    # --- 正常短信 (Ham) ---
    "妈 我 今晚 不 回家 吃饭 了 你们 先 吃",
    "李总 下午 两点 的 会议 请 准时 参加 附件 是 报表",
    "亲爱 的 客户 您的 快递 已 放入 丰巢 柜 请 及时 取件",
    "周末 去 哪里 玩 听说 那个 新 开 的 商场 不错",
    "老师 您好 我是 张三 这是 我 本周 的 实验 报告 请 查收",
    "晚上 一起 峡谷 见 我 打 野 你 辅助",
    "明天 降温 记得 多 穿 点 衣服 别 感冒 了",
    "好的 收到 谢谢 领导",
    # --- 欺诈/垃圾短信 (Spam/Fraud) ---
    "尊敬 的 用户 您的 银行卡 账户 存在 异常 请 点击 链接 验证 否则 冻结",
    "恭喜 您 手机 号码 被 抽中 苹果 电脑 一台 请 拨打 电话 领取",
    "澳门 线上 赌场 注册 即 送 百万 现金 点击 链接 马上 提现",
    "内部 消息 某某 股票 明天 涨停 加 微信 免费 领取 暴富 秘籍",
    "您的 信用卡 额度 已经 提升 至 10万 请 点击 链接 激活",
    "无抵押 贷款 极速 放款 不看 征信 点击 链接 申请",
    "高薪 诚聘 兼职 打字员 一天 赚 500 块 详情 加 扣扣"
]

# 0 代表正常短信，1 代表欺诈短信
labels = [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1]

df = pd.DataFrame({
    '短信内容': texts,
    '是否欺诈': labels
})

print(f'数据集大小: {df.shape}')
df.head(10)

<hr><hr>

#### 第二步：查看数据的基本信息（数据探查 EDA）
<hr>

In [ ]:
# 把标签转成汉字，方便统计
df['类型'] = df['是否欺诈'].map({0: '正常', 1: '欺诈'})

label_count = df['类型'].value_counts()
print("短信类型分布：\n", label_count)

In [ ]:
import matplotlib.font_manager as fm
zh_fonts = [f.name for f in fm.fontManager.ttflist 
            if any(kw in f.name for kw in ['Hei', 'Song', 'CJK', 'Chinese', 'SC', 'TC', 'Gothic', 'SimHei'])]
if zh_fonts:
    plt.rcParams['font.family'] = zh_fonts[0]
plt.rcParams['axes.unicode_minus'] = False 

plt.figure(figsize=(6, 5))
plt.pie(x=label_count, labels=label_count.index, autopct='%.1f%%', 
        colors=['lightblue', 'lightcoral'], explode=[0, 0.1], radius=0.8)
plt.title('欺诈短信 vs 正常短信 比例分布')
plt.show()

<hr><hr>

#### 第三步：极其关键的“文本向量化” (Text Vectorization)
<hr>

计算机是个“文盲”，它连“银行卡”这三个字都不认识，它只懂矩阵和数字。
怎么把一句话变成一堆数字？这就需要**特征提取（文本向量化）**。

In [ ]:
# 方法 1：词袋模型 (Bag of Words / CountVectorizer)
# 这是最简单粗暴的方法。它做两件事：
# 1. 把所有短信里的词汇总起来，编一本字典。
# 2. 数一数每一条短信里，字典里的每个词出现了几次。

# 创建词袋模型
count_vec = CountVectorizer()
X_counts = count_vec.fit_transform(df['短信内容'])

print("词袋模型发现的全部单词（字典）：")
vocab = count_vec.get_feature_names_out()
print(vocab)
print(f"\n字典里总共有 {len(vocab)} 个不同的词。")

print("\n我们把第一条短信翻译给机器听，它变成了这样：")
print("原短信：", df['短信内容'][0])
print("词频矩阵：\n", X_counts.toarray()[0])

# 看看那些非 0 的数字，它们分别对应着字典里的某个位置，代表这个词在这句话里出现了 1 次。
# 这就是词袋模型：把一句话变成了一串长长的 0 和 1（或更多）的数组。

<hr>

#### 第四步：模型训练
<hr>

In [ ]:
# 把刚才生成的特征矩阵 X 和标签 y 拿出来
X = X_counts
y = df['是否欺诈'].values

# 切分考卷
X_train, X_test, y_train, y_test = model_selection.train_test_split(
    X, y, 
    random_state=42, 
    test_size=0.3
)

# 召唤主角：多项式朴素贝叶斯模型 (MultinomialNB)
# 注意：上一章判断良性/恶性肿瘤，因为特征是具体的数值（厚度1~10），用的是高斯贝叶斯 (GaussianNB)。
# 现在处理的是文本词频（出现了 0 次、1 次、2 次...离散整数），所以必须用多项式贝叶斯！
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

print("贝叶斯模型训练完毕，它已经记住了欺诈分子的常用话术概率！")

<hr><hr>

#### 第五步：模型评估与预测
<hr>

In [ ]:
# 做期末考试卷
y_pred = nb_model.predict(X_test)

# 计算准确率
acc = accuracy_score(y_test, y_pred)
print(f"贝叶斯分类器的准确率 = {(acc * 100):.2f}%")

In [ ]:
# 实战演练：抓取两根“漏网之鱼”的新短信来测试
test_sms = [
    "亲爱 的 用户 点击 链接 即可 免费 领取 游戏 礼包 账户 绝对 安全", # 故意混入“亲爱的”、“安全”这种好词
    "明天 下午 三点 记得 来 办公室 签合同"
]

# 必须用刚才建好的词袋字典 (count_vec) 来转换新短信！千万不能重新 fit！
test_sms_counts = count_vec.transform(test_sms)

pred_results = nb_model.predict(test_sms_counts)
label_names = ["正常短信", "🚨 欺诈警告"]

print("\n【人工智能防诈骗雷达 启动】")
for i, msg in enumerate(test_sms):
    print(f"[{label_names[pred_results[i]]}] -> 拦截内容: {msg}")

<hr><hr>

#### 第六步：【拓展提高】词袋模型的致命缺陷与 TF-IDF 降维打击
<hr>

刚才的词袋模型（CountVectorizer）很好理解：哪个词出现得多，分数就高。
但你仔细想想，语言世界里有一个巨大的 BUG！

在汉语里，“的”、“了”、“是”、“请”这种词，无论是正常短信还是欺诈短信，它们出现的频率绝对是最高的。
按照词袋模型的逻辑，机器一看短信里有个“的”字，记 1 分。有个“点击”记 1 分。
机器会觉得：“的”字和“点击”一样重要！
这简直是胡扯。“的”字没有任何分类价值，纯属废话；而“点击”、“链接”、“免费”才是抓住骗子的命门！

**如何让机器学会区分“废话”和“关键词”？**

答案是：**TF-IDF（词频-逆文档频率）**。这也许是自然语言处理历史上最伟大、最优雅的数学公式。

In [ ]:
# 我们不再用 CountVectorizer，改用更高级的 TfidfVectorizer
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(df['短信内容'])

# 我们把两种方法算出来的值对比一下
word_index = list(vocab).index("请")
click_index = list(vocab).index("点击")

print("【普通词频 (Count) 眼里的世界】")
print(f"'请' 出现的总次数: {X_counts[:, word_index].sum()}")
print(f"'点击' 出现的总次数: {X_counts[:, click_index].sum()}")
print("模型：嗯，它们两个一样重要。\n")

print("【TF-IDF 魔法加持下的世界】")
print(f"'请' 的总权重得分: {X_tfidf[:, word_index].sum():.4f}")
print(f"'点击' 的总权重得分: {X_tfidf[:, click_index].sum():.4f}")
print("模型：破案了！'点击' 这种专业词汇的含金量，远大于烂大街的 '请' 字！")

##### TF-IDF 到底是怎么算出来的？

它的名字拆开看：**TF $\times$ IDF**。

1. **TF (Term Frequency，词频)**：
一条短信里，某个词出现的次数越多，这个词对这条短信越重要。（比如短信里说了三个“免费”，那绝对在搞推销）。
*结论：在单条短信内部，次数越多越重要。*

2. **IDF (Inverse Document Frequency，逆文档频率)**：
核心杀招来了。机器去翻看这 15 条短信的语料库，它发现“的”、“请”字在 14 条短信里都出现了。
物以稀为贵！一个词在所有文档里都出现，说明它是毫无特点的“口水词”。相反，“银行卡”只在 1 条短信里出现过，说明它极具特色！

**数学公式：** $\text{IDF} = \log \left( \frac{\text{文档总数}}{\text{包含该词的文档数}} \right)$

如果“的”字在所有文档都出现，分母和分子一样大，除出来是 1，$\log(1) = 0$。
“的”字的 IDF 权重直接变成 0，被当场抹杀！

**TF-IDF 就是在寻找那些：“在这篇文章里反复出现，但在其他文章里极少出现” 的灵魂词汇。**
以后再做文本分类，果断扔掉 CountVectorizer，无脑上 TF-IDF 就对了！

<hr><hr>

#### 第七步：【拓展提高】上帝掷骰子 —— 贝叶斯定理的侦探逻辑
<hr>

我们搞懂了怎么把文本变成数字。最后，我们要搞懂，那个叫做 `MultinomialNB()` 的模型，拿到这些数字后，在脑子里到底算了什么。

很多人一看到“朴素贝叶斯”公式，脑子直接宕机：
$$ P(A|B) = \frac{P(B|A) \times P(A)}{P(B)} $$

别怕，把公式扔掉。我们来玩一局侦探游戏。

##### 1. 案发现场
警察（模型）截获了一条新短信，里面只有一个关键词：“**转账**”。
警察要判断：这条短信是欺诈（Spam）的概率有多大？
也就是说，我们要求解：**$P(\text{欺诈} | \text{转账})$**。（在看到“转账”这个线索的前提下，它是欺诈的概率）。

警察不认识字，他只能去查过去的“案底”（训练集）。

##### 2. 第一步：摸底（先验概率 Prior）
警察先不去管“转账”这俩字。他先查案底：在咱们这个地区，平时发短信，到底有多少是骗子？
假设一共 100 条短信，有 20 条是骗子发的，80 条是正常人发的。
那么，在没有任何线索之前，警察认定任何一条短信是欺诈的概率是 20%。
这叫**先验概率 $P(\text{欺诈}) = 0.2$**。（基于历史经验的底色）。

##### 3. 第二步：分析作案手法（似然度 Likelihood）
警察开始专门研究那 20 条已经被抓起来的骗子发出的短信。
他发现，骗子发出的短信里，有 10 条都包含“转账”这两个字。
那么，**如果对方真的是个骗子**，他嘴里说出“转账”的概率是多少？是 10/20 = 50%。
这叫**似然度 $P(\text{转账} | \text{欺诈}) = 0.5$**。（骗子的作案特征）。

同理，他又去查那 80 条正常短信。发现正常人偶尔也给人转账（比如还钱），有 4 条正常短信包含“转账”。
那么，**如果对方是个正常人**，他说出“转账”的概率是 4/80 = 5%。

##### 4. 第三步：案情收网（后验概率 Posterior）
好了，一切准备就绪。现在警察手里拿着这条带有“转账”的未知短信，开始用贝叶斯大脑飞速计算：

这条短信有多大的“作恶分值”？
作恶分值 = （它是骗子的底色概率） $\times$ （骗子爱说“转账”的概率）
$= 0.2 \times 0.5 = 0.1$

这条短信有多大的“清白分值”？
清白分值 = （它是好人的底色概率） $\times$ （好人说“转账”的概率）
$= 0.8 \times 0.05 = 0.04$

最后，警察在心里掂量一下这两个分值：
它是欺诈的真实概率 = $\frac{\text{作恶分值}}{\text{作恶分值} + \text{清白分值}}$
$= \frac{0.1}{0.1 + 0.04} = \frac{0.1}{0.14} \approx 71.4\%$

最终结论：在看到“转账”这个词后，结合当地治安底色和骗子话术习惯，警察判定，这条短信有 71.4% 的概率是诈骗！

**这就是贝叶斯定理！**
公式里的 $P(A|B) = \frac{P(B|A) \times P(A)}{P(B)}$，其实就是：
**最终判定概率 = $\frac{\text{作恶分值}}{\text{所有人说出这个词的总概率}}$**

##### 为什么叫“朴素(Naive)”贝叶斯？
刚才我们只用了一个词“转账”。如果短信里有三个词“转账”、“安全”、“免费”。
真实的数学中，这三个词可能会互相影响（比如骗子说了转账就爱说安全）。
但贝叶斯模型是个一根筋的直男，它极其“朴素（天真）”地认为：**所有词之间都是毫无关联、各自独立的！**
它直接把三个词的概率简单粗暴地乘在一起。虽然这个假设在现实中很可笑，但诡异的是，这种粗暴的方法在反垃圾邮件、欺诈识别中，效果出奇的好，而且计算速度快到不可思议！

#### 总结

| 概念 | 大白话解释 | 在欺诈识别中的作用 |
|------|------|------|
| **词袋模型 (CountVectorizer)** | 查字典，数次数。 | 把中文句子翻译成包含 0 和 1 的数学矩阵，这是计算机处理语言的第一步。 |
| **TF-IDF 向量化** | 找出在一句话里常出现，但在全网罕见的词。 | 降维打击，自动过滤掉“的”、“是”这种废话，给“链接”、“提现”这种破案关键词极高的权重。 |
| **先验概率 (Prior)** | 案发前的底色。 | 大盘数据里有多少比例是骗子。 |
| **似然度 (Likelihood)** | 骗子的作案习惯。 | 骗子说出某句话的概率。 |
| **朴素贝叶斯 (Naive Bayes)** | 拿着历史案底和作案习惯，反推眼前这条短信的作恶概率。 | 完全依靠概率统计，没有复杂的网络结构，对大规模文本分类速度极快，防骚扰/反诈雷达的底层核心。 |

<br>

> **关键点**：今天你不仅完成了一个反欺诈引擎，你还真正理解了机器是如何“阅读”文字的（TF-IDF）。你会发现，所谓的人工智能判断真假，其实就是在计算：**“如果它是骗子，它这样说话的概率有多大？”**

<br>

<hr><hr>

## 实验五完成
<hr>

##### 此实验教材最近更新时间 2026年3月17日
<hr><hr>